## The model is trained on the Intel Dataset from keggle competetion , to classify the picture in the groups : buildings, forest, glacier, mountain, sea, street

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'intel-image-classification' dataset.
Path to dataset files: /kaggle/input/intel-image-classification


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Flatten,Conv2D,MaxPooling2D,Dropout,BatchNormalization

First, let's define the paths for our training and testing data. The `path` variable was already created from the dataset download.

In [ ]:
import os

train_dir = os.path.join(path, 'seg_train', 'seg_train')
test_dir = os.path.join(path, 'seg_test', 'seg_test')

print(f"Training data directory: {train_dir}")
print(f"Testing data directory: {test_dir}")

Training data directory: /kaggle/input/intel-image-classification/seg_train/seg_train
Testing data directory: /kaggle/input/intel-image-classification/seg_test/seg_test


Now, we'll use `tf.keras.utils.image_dataset_from_directory` to load the training data and create a validation split. This function automatically infers labels from the directory names and handles batching and resizing. We'll set a `validation_split` to allocate a portion of the training data for validation.

In [ ]:
IMG_WIDTH = 150
IMG_HEIGHT = 150
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2, # 20% of data for validation
    subset='training',
    seed=42 # for reproducible split
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset='validation',
    seed=42
)

# Get class names from the training dataset
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nClasses found: {class_names}")
print(f"Number of classes: {num_classes}")

# Print shapes of a batch to verify
for images, labels in train_ds.take(1):
    print(f"Shape of a batch of training images: {images.shape}")
    print(f"Shape of a batch of training labels: {labels.shape}")

Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.

Classes found: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Number of classes: 6
Shape of a batch of training images: (32, 150, 150, 3)
Shape of a batch of training labels: (32,)


In [ ]:
model = Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(IMG_HEIGHT,IMG_WIDTH,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(IMG_HEIGHT,IMG_WIDTH,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(IMG_HEIGHT,IMG_WIDTH,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(256,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(IMG_HEIGHT,IMG_WIDTH,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(256,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(6,activation='sigmoid'))

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 148, 148, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 72, 72, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 34, 34, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 15, 15, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 15, 15, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │     3,211,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,643,398 (13.90 MB)

 Trainable params: 3,642,438 (13.89 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(train_ds,epochs=10,validation_data=val_ds)

Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 55s 121ms/step - accuracy: 0.5292 - loss: 1.3815 - val_accuracy: 0.5296 - val_loss: 1.2105
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 16s 46ms/step - accuracy: 0.6909 - loss: 0.8638 - val_accuracy: 0.5727 - val_loss: 1.2103
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.7411 - loss: 0.7301 - val_accuracy: 0.6771 - val_loss: 0.8494
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 23s 47ms/step - accuracy: 0.7742 - loss: 0.6241 - val_accuracy: 0.6989 - val_loss: 0.8193
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.7879 - loss: 0.5605 - val_accuracy: 0.7391 - val_loss: 0.5993
Epoch 6/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.8007 - loss: 0.5035 - val_accuracy: 0.6750 - val_loss: 0.8038
Epoch 7/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.8105 - loss: 0.4774 - val_accuracy: 0.7687 - val_loss: 0.5706
Epoch 8/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.8251 - loss: 0.4099 -

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2

def ans(img):
  test_img = cv2.imread(img)
  # plt.imshow(test_img)
  test_img = cv2.resize(test_img,(150,150))
  test_input = test_img.reshape((1,150,150,3))

  classes = ['buildings','forest','glacier','mountain','sea','street']

  pred = np.argmax(model.predict(test_input,verbose=False))
  print(classes[pred])

In [ ]:
ans('/content/images-2.jpeg')
ans('/content/forest-path-v3-13197528.jpg.webp')
ans('/content/GlacierBlues_Header3.jpg.webp')
ans('/content/mt-everest.jpg.avif')
ans('/content/beautiful-beach-tropical-beach-background-as-summer-landscape-white-sand-and-calm-sea-for-beach-banner-perfect-beach-scene-vacation-and-summer-holiday-concept-boost-up-color-process-photo.jpg')
ans('/content/istockphoto-1628010210-612x612.jpg')

buildings
forest
glacier
mountain
sea
street


In [ ]:
from keras.applications.vgg16 import VGG16

vgg = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(150,150,3)
)